# Review Uncertain Charts

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 1.0.0 | 2026-01-28 | That Le | Review and reclassify uncertain charts |

---

## Purpose

Review charts with confidence < 70% and:
1. Identify false positives (not charts)
2. Manually assign correct type
3. Prepare clean data for model re-training

In [ ]:
# ============================================================================
# SETUP
# ============================================================================
import sys
from pathlib import Path
import json
import shutil
import random

PROJECT_ROOT = Path.cwd().parent
UNCERTAIN_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "classified_charts" / "uncertain"
REVIEW_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "review"

# Categories for review
FALSE_POSITIVE_DIR = REVIEW_DIR / "false_positives"  # Not charts
RECLASSIFY_DIR = REVIEW_DIR / "reclassify"  # Need manual classification

REVIEW_DIR.mkdir(parents=True, exist_ok=True)
FALSE_POSITIVE_DIR.mkdir(parents=True, exist_ok=True)
RECLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

uncertain_images = list(UNCERTAIN_DIR.glob("*.png"))
print(f"Uncertain charts: {len(uncertain_images):,}")

In [ ]:
# ============================================================================
# SAMPLE UNCERTAIN CHARTS
# ============================================================================
from PIL import Image
import matplotlib.pyplot as plt

# Random sample
samples = random.sample(uncertain_images, min(16, len(uncertain_images)))

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis("off")

plt.suptitle("Sample Uncertain Charts (< 70% confidence)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# LOAD CLASSIFICATION RESULTS
# ============================================================================

results_file = PROJECT_ROOT / "data" / "academic_dataset" / "classified_charts" / "classification_results.json"

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    
    # Filter uncertain
    uncertain_results = [r for r in results["results"] if r["dest"] == "uncertain"]
    
    # Analyze confidence distribution
    confidences = [r["confidence"] for r in uncertain_results]
    
    print(f"Uncertain charts: {len(uncertain_results):,}")
    print(f"Confidence range: {min(confidences):.2%} - {max(confidences):.2%}")
    print(f"Mean confidence: {sum(confidences)/len(confidences):.2%}")
    
    # Distribution by predicted type
    print("\nPredicted types (uncertain):")
    type_counts = {}
    for r in uncertain_results:
        t = r["type"]
        type_counts[t] = type_counts.get(t, 0) + 1
    
    for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {t:12s}: {c:,}")
else:
    print("No classification results file found.")

In [ ]:
# ============================================================================
# IDENTIFY FALSE POSITIVES (Heuristics)
# ============================================================================
from PIL import Image
import numpy as np


def analyze_image(img_path):
    """Basic heuristics to detect false positives."""
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img)
    
    # Image dimensions
    h, w = arr.shape[:2]
    aspect = w / h
    
    # Color analysis
    gray = np.mean(arr, axis=2)
    std = np.std(gray)
    
    # Detect mostly text (high contrast, small std)
    is_text_heavy = std > 80
    
    # Detect very small images
    is_small = w < 100 or h < 100
    
    # Detect extreme aspect ratios (likely banners/headers)
    is_banner = aspect > 5 or aspect < 0.2
    
    return {
        "width": w,
        "height": h,
        "aspect": aspect,
        "std": std,
        "likely_false_positive": is_small or is_banner,
    }


# Analyze sample
print("Analyzing uncertain images...")
likely_fp = []

for img_path in uncertain_images[:500]:  # Sample first 500
    try:
        analysis = analyze_image(img_path)
        if analysis["likely_false_positive"]:
            likely_fp.append((img_path, analysis))
    except Exception as e:
        pass

print(f"Likely false positives (from 500 sample): {len(likely_fp)}")

In [ ]:
# ============================================================================
# SHOW LIKELY FALSE POSITIVES
# ============================================================================

if likely_fp:
    samples = likely_fp[:16]
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    axes = axes.flatten()
    
    for ax, (img_path, analysis) in zip(axes, samples):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f"{analysis['width']}x{analysis['height']}\naspect={analysis['aspect']:.1f}", fontsize=9)
        ax.axis("off")
    
    for ax in axes[len(samples):]:
        ax.axis("off")
    
    plt.suptitle("Likely False Positives (small/banner images)", fontsize=14)
    plt.tight_layout()
    plt.show()

---

## Manual Review Actions

After reviewing, you can:

1. **Move false positives** to `review/false_positives/`
2. **Move for reclassification** to `review/reclassify/`
3. **Use cleaned data** to re-train models

### Re-training Strategy

| Model | Current Data | New Data |
| --- | --- | --- |
| YOLO | 2,575 charts | + ~40k verified charts |
| ResNet | 2,575 charts | + ~40k with types |

In [ ]:
# ============================================================================
# SUMMARY
# ============================================================================

print("=" * 60)
print("REVIEW SUMMARY")
print("=" * 60)
print(f"\nUncertain charts: {len(uncertain_images):,}")
print(f"Likely false positives (heuristic): ~{len(likely_fp) * len(uncertain_images) // 500:,}")
print(f"\nRecommendation:")
print("1. Filter out small/banner images automatically")
print("2. Sample 500-1000 for manual review")
print("3. Use ~40k high-confidence charts for re-training")
print("=" * 60)